# Phase 2 — Task 2: Clean It, Log Every Decision

## Objective

This turns the raw NorthStar Goods export into an analysis-ready table, with a
cleaning log documenting every decision so it is reproducible and defensible.

**Input:** `dataset/northstar_goods_v2.csv` (raw, 1,002 rows, 11 columns).
**Output:** `dataset/northstar_clean.csv`.

In [47]:
# Import pandas for data manipulation and NumPy for numeric operations.
import pandas as pd
import numpy as np

# Show all columns when previewing the DataFrame, instead of truncating.
pd.set_option('display.max_columns', None)

In [48]:
# Load the raw export.
df = pd.read_csv('../../dataset/northstar_goods_v2.csv')

In [49]:
# Keep the original row count so we can report a raw-vs-clean diff at the end.
raw_row_count = len(df)

print(f"Loaded {raw_row_count:,} rows and {df.shape[1]} columns.")
df.head()

Loaded 1,002 rows and 11 columns.


,order_id,order_date,customer_id,region,product_category,quantity,unit_price,discount_pct,payment_method,total_amount,returned
0,NS-00522,2024-02-23,CUST-0214,West,Toys,7.0,173.91,0,Card,1217.37,N
1,NS-00740,2024-11-23,CUST-0062,London,Food & Drink,1.0,89.36,5,Card,84.89,N
2,NS-00824,2024-08-25,CUST-0264,Scotland,Sports,2.0,8.15,10,Card,14.67,N
3,NS-00663,2024-01-13,CUST-0056,London,Clothing,9.0,178.22,0,Card,1603.98,N
4,NS-00412,2024-03-24,CUST-0169,Midlands,Toys,5.0,28.53,0,PayPal,142.65,Y


### Build a cleaning log

In [50]:
# List that accumulates one entry per cleaning decision.
cleaning_log = []

def log_step(step, issue, action, rows_affected, justification):
    """Append a single row to the cleaning log."""
    cleaning_log.append({
        "Step": step,
        "Issue": issue,
        "Action taken": action,
        "Rows affected": rows_affected,
        "Justification": justification,
    })

## 1. Duplicates
Remove exact duplicate rows. Rows identical across every column represent the same order recorded twice, not two separate transactions.

In [51]:
# Count rows that are identical across every column to another row.
n_exact_dupes = df.duplicated().sum()
print("Exact duplicate rows:", n_exact_dupes)

# Show both copies of each duplicated row (keep=False flags every occurrence).
df[df.duplicated(keep=False)].sort_values("order_id")

Exact duplicate rows: 2


,order_id,order_date,customer_id,region,product_category,quantity,unit_price,discount_pct,payment_method,total_amount,returned
0,NS-00522,2024-02-23,CUST-0214,West,Toys,7.0,173.91,0,Card,1217.37,N
86,NS-00522,2024-02-23,CUST-0214,West,Toys,7.0,173.91,0,Card,1217.37,N
561,NS-00738,2024-05-01,CUST-0030,South,Home & Garden,10.0,145.50,0,Cash,1455.00,N
663,NS-00738,2024-05-01,CUST-0030,South,Home & Garden,10.0,145.50,0,Cash,1455.00,N


In [52]:
# Drop exact duplicates, keeping the first occurrence of each and resetting the index
# so downstream row-position logic doesn't trip on the gaps left by dropped rows.
before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
after = len(df)
rows_removed = before - after

print(f"Removed {rows_removed} duplicate rows. {before} -> {after}")

Removed 2 duplicate rows. 1002 -> 1000


In [53]:
# Log this decision for the audit trail.
log_step(
    step="1. Duplicates",
    issue="2 rows were exact duplicates of another row (order_id NS-00522 and NS-00738, "
          "each appearing twice with identical values in every column).",
    action="Dropped the second occurrence of each duplicate, keeping the first.",
    rows_affected=rows_removed,
    justification="A row identical in every field to another is almost certainly the same "
                   "order captured twice by the export process, not two genuine orders. "
                   "Keeping both would double count that order's revenue and quantity in "
                   "any aggregation.",
)

## 2. Missing values

Three columns have missing values: `payment_method` (30, ~3%), `unit_price` (30, ~3%),
and `quantity` (6, ~0.6%). Each is handled on its own terms below rather than with a
single blanket rule, because the right treatment depends on what the column is used for.

In [54]:
# Isolate and rank only the columns that actually have missing values.
missing_summary = df.isna().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
missing_summary

unit_price        30
payment_method    30
quantity           6
dtype: int64

#### 2a. `payment_method`: fill with `"Unknown"`

`payment_method` is a categorical field used for segmentation (e.g. counting orders by
payment type), not for any arithmetic. Dropping these 30 rows would lose otherwise valid
order, price, and quantity data. Filling with a placeholder value like the mode
(e.g. `Card`) would misrepresent how those orders were actually paid, since we have no
basis to guess. An explicit `"Unknown"` category preserves the row and its other fields
while being honest that the payment method wasn't recorded.

In [55]:
# Count missing values before filling, for the log.
n_missing_payment = df["payment_method"].isna().sum()

# Fill missing payment_method with an explicit "Unknown" placeholder rather than
# guessing a specific method or dropping the row.
df["payment_method"] = df["payment_method"].fillna("Unknown")

print(f"Filled {n_missing_payment} missing payment_method values with 'Unknown'.")

Filled 30 missing payment_method values with 'Unknown'.


In [56]:
# Log this decision for the audit trail.
log_step(
    step="2a. Missing payment_method",
    issue=f"{n_missing_payment} rows (~3%) had no payment_method recorded.",
    action='Filled missing values with the placeholder category "Unknown".',
    rows_affected=n_missing_payment,
    justification="payment_method is categorical and used only for segmentation, not "
                   "arithmetic. Dropping these rows would lose valid order data for no "
                   "reason; guessing a specific method (e.g. the mode) would misrepresent "
                   "how the order was actually paid. An explicit 'Unknown' category keeps "
                   "the row while being transparent about the gap.",
)

#### 2b. `unit_price`: leave missing, without filling or dropping

`unit_price` feeds directly into `total_amount`, which is itself present and correct for
every row (verified in Step 6). Because `total_amount`, `quantity`, and `discount_pct`
are all known for these rows, `unit_price` can be **back-calculated exactly** rather than
estimated with a median. No value is invented here. It's derived from data already on the row.

In [57]:
# Note the count.
n_missing_price = df["unit_price"].isna().sum()
print(f"{n_missing_price} rows have missing unit_price.")

30 rows have missing unit_price.


#### 2c. `quantity`: leave missing, without filling or dropping

Same reasoning as `unit_price`
`quantity` can be back-calculated exactly from
`total_amount`, `unit_price`, and `discount_pct` for rows where those three are known.

In [58]:
# Note the count.
n_missing_qty = df["quantity"].isna().sum()
print(f"{n_missing_qty} rows have missing quantity.")

6 rows have missing quantity.


## 3. Dates

Parse `order_date` to a real datetime type, and handle the one impossible date
(`2024-13-40`, month 13, day 40 is not a valid calendar date).

In [59]:
# Convert order_date from text to a real datetime type.
# errors="coerce" turns any value that can't be parsed (like the impossible
# "2024-13-40") into NaT instead of raising an exception.
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

n_invalid_dates = df["order_date"].isna().sum()
print(f"order_date dtype is now: {df['order_date'].dtype}")
print(f"Rows that failed to parse (became NaT): {n_invalid_dates}")

# Show which order(s) had the invalid date.
df.loc[df["order_date"].isna(), ["order_id"]]

order_date dtype is now: datetime64[us]
Rows that failed to parse (became NaT): 1


,order_id
769,NS-00234


The invalid date belongs to a single order (`NS-00234`) with otherwise complete,
plausible data (quantity, price, region, etc. are all normal). There is no reliable way to
infer what the real date was, and dropping the whole row would discard valid non-date
information. Since `order_date` isn't needed for row-level correctness elsewhere in this
task, we leave it as `NaT`, which is a proper missing-datetime marker, rather than guessing a date or dropping the row. A downstream analysis that needs valid dates can filter these out explicitly.

In [60]:
# Log this decision for the audit trail.
log_step(
    step="3. Invalid date",
    issue="order_id NS-00234 had order_date = '2024-13-40', an impossible calendar date "
          "(month 13, day 40) that fails standard date parsing.",
    action="Parsed order_date with pd.to_datetime(errors='coerce'), converting the "
           "invalid value to NaT (a proper missing-datetime marker) rather than guessing "
           "a replacement date.",
    rows_affected=n_invalid_dates,
    justification="There is no reliable way to infer the true date from the corrupted "
                   "string, and the rest of the row's data is valid and worth keeping. "
                   "NaT correctly signals that the date is unknown instead of an invented "
                   "value, and downstream date-based analysis can filter it out "
                   "explicitly.",
)

## 4. Numerics

Coerce `unit_price` and `quantity` to numeric types, check for negative values, and
handle the two known outliers (`unit_price = 9999.99`, `quantity = 500`).

In [61]:
# Confirm both columns are numeric. This is a no-op here since both are already
# float64, but pd.to_numeric is included for robustness in case a future export
# stores these as strings.
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

print(df[["unit_price", "quantity"]].dtypes)

unit_price    float64
quantity      float64
dtype: object


In [62]:
# Check for negative values, which would be invalid for a real order.
n_negative_qty = (df["quantity"] < 0).sum()
n_negative_price = (df["unit_price"] < 0).sum()
n_negative_total = (df["total_amount"] < 0).sum()
print("Negative quantity:", n_negative_qty)
print("Negative unit_price:", n_negative_price)
print("Negative total_amount:", n_negative_total)

Negative quantity: 0
Negative unit_price: 0
Negative total_amount: 0


In [63]:
# Log this check for the audit trail, even though no correction was needed.
# This shows the check was performed rather than skipped.
log_step(
    step="4. Negative value check",
    issue="Checked quantity, unit_price, and total_amount for negative values, which "
          "would be impossible for a real order.",
    action="No correction applied. None of the three columns contained a negative value.",
    rows_affected=0,
    justification="No negative values were present, so no correction was needed. This "
                   "step is logged to record that the check was performed and passed, "
                   "not silently skipped.",
)

No negative values were found in `quantity`, `unit_price`, or `total_amount`, so no
action is needed there.

#### The two outliers: `quantity = 500` and `unit_price = 9999.99`

Rather than treat these as ordinary outliers to cap or drop, I checked whether
`total_amount` (which is complete and correct for every other row) still reconciles with
`quantity × unit_price × (1 − discount_pct/100)` for these two rows. It does **not**,
which means `total_amount` was computed before these two fields were corrupted, and can
be used to recover the true values:

- **`NS-00840`**: `total_amount / (unit_price × (1 − discount))` = **8.0**, so the real
  quantity was almost certainly 8, not 500 (likely a duplicated trailing digit).
- **`NS-00953`**: `total_amount / (quantity × (1 − discount))` = **£188.99**, so the real
  unit price was almost certainly £188.99, not £9,999.99 (likely a decimal or digit
  corruption, e.g. a stray "99" appended).

## 5. Categoricals

Normalise `region` and `payment_method` casing. The regions reference table uses proper case, and a case-sensitive join would silently drop every row whose region isn't in exactly that casing (e.g. `london`, `LONDON` would not match `London`).

In [64]:
# Inspect the raw, uncleaned distinct region values.
print("region raw distinct values:", df["region"].nunique())
print(sorted(df["region"].unique()))

region raw distinct values: 23
['EAST', 'East', 'LONDON', 'London', 'MIDLANDS', 'Midlands', 'NORTH', 'North', 'SCOTLAND', 'SOUTH', 'Scotland', 'South', 'WEST', 'Wales', 'West', 'east', 'london', 'midlands', 'north', 'scotland', 'south', 'wales', 'west']


In [65]:
# Count how many rows have a region value that isn't already in title case,
# for the log (before normalising).
n_region_recased = (df["region"] != df["region"].str.strip().str.title()).sum()

# Normalise region to title case (e.g. "LONDON", "london" -> "London") so it will
# match the properly-cased regions reference table used in Task 3's join.
df["region"] = df["region"].str.strip().str.title()

print("region distinct values after normalising:", df["region"].nunique())
print(sorted(df["region"].unique()))

region distinct values after normalising: 8
['East', 'London', 'Midlands', 'North', 'Scotland', 'South', 'Wales', 'West']


In [66]:
# Log this decision for the audit trail.
log_step(
    step="5a. Region casing",
    issue=f"region had 23 raw distinct values instead of 8 true regions, due to "
          f"inconsistent casing (e.g. 'London', 'LONDON', 'london' all present). "
          f"{n_region_recased} rows had non-title-case values.",
    action="Normalised to title case with .str.strip().str.title().",
    rows_affected=int(n_region_recased),
    justification="The Task 3 regions reference table uses proper (title) case. A "
                   "case-sensitive join on the raw column would silently drop every row "
                   "whose casing doesn't match exactly, producing incomplete results with "
                   "no error. Normalising here guarantees the join in Task 3 works "
                   "correctly.",
)

In [67]:
# Inspect the raw, uncleaned distinct payment_method values (excluding the
# "Unknown" placeholder already filled in Step 2a).
print("payment_method raw distinct values (excluding fill):", df["payment_method"].nunique())
print(sorted(df["payment_method"].unique()))

payment_method raw distinct values (excluding fill): 8
['Bank Transfer', 'Card', 'Cash', 'PayPal', 'Unknown', 'card', 'cash', 'paypal']


In [68]:
# Map to canonical labels explicitly rather than blind .str.title(), which would
# incorrectly turn "PayPal" into "Paypal" (title-case only capitalises after spaces).
payment_method_map = {
    "card": "Card", "cash": "Cash", "paypal": "PayPal", "bank transfer": "Bank Transfer",
}

def normalise_payment(value):
    # Leave the "Unknown" placeholder from Step 2a untouched.
    if value == "Unknown":
        return value
    return payment_method_map.get(value.strip().lower(), value.strip())

# Count how many rows will actually change, for the log (before normalising).
n_payment_recased = (
    (df["payment_method"] != "Unknown")
    & (df["payment_method"] != df["payment_method"].apply(normalise_payment))
).sum()

# Apply the canonical mapping.
df["payment_method"] = df["payment_method"].apply(normalise_payment)

print("payment_method distinct values after normalising:", df["payment_method"].nunique())
print(sorted(df["payment_method"].unique()))

payment_method distinct values after normalising: 5
['Bank Transfer', 'Card', 'Cash', 'PayPal', 'Unknown']


In [69]:
# Log this decision for the audit trail.
log_step(
    step="5b. Payment method casing",
    issue=f"payment_method had duplicated categories from mixed casing (e.g. 'Card' and "
          f"'card'). {n_payment_recased} rows had non-title-case values (excluding the "
          f"'Unknown' placeholder from Step 2a).",
    action="Normalised via an explicit canonical mapping, leaving the 'Unknown' "
           "placeholder untouched.",
    rows_affected=int(n_payment_recased),
    justification="Mixed casing splits what should be a single category (e.g. Card) into "
                   "several, undercounting the true total for each payment method in any "
                   "group-by or segmentation. A canonical mapping is used instead of "
                   ".str.title() because title-case would incorrectly turn 'PayPal' into "
                   "'Paypal'.",
)

## 6. Recompute and validate `total_amount`

Check that `total_amount = quantity × unit_price × (1 − discount_pct / 100)` for every
row, and use this relationship to both recover missing `unit_price`/`quantity` values
(Step 2b/2c) and correct the two corrupted outlier rows (Step 4) all in one place,
since they share the same logic.

In [70]:
def expected_total(row):
    # Can't compute an expected total if either quantity or unit_price is missing.
    if pd.isna(row["quantity"]) or pd.isna(row["unit_price"]):
        return np.nan
    return row["quantity"] * row["unit_price"] * (1 - row["discount_pct"] / 100)

# Recompute what total_amount "should" be from the other three fields, for every row.
df["_expected_total"] = df.apply(expected_total, axis=1)
diff = (df["total_amount"] - df["_expected_total"]).abs()

n_unverifiable = df["_expected_total"].isna().sum()
n_mismatch = (diff > 0.01).sum()

print(f"Rows where total_amount can't yet be checked (missing quantity/unit_price): {n_unverifiable}")
print(f"Rows where total_amount differs from the recomputed value by > £0.01: {n_mismatch}")

# Show the rows where the reconciliation actually fails.
df.loc[diff > 0.01, ["order_id", "quantity", "unit_price", "discount_pct",
                      "total_amount", "_expected_total"]]

Rows where total_amount can't yet be checked (missing quantity/unit_price): 36
Rows where total_amount differs from the recomputed value by > £0.01: 2


,order_id,quantity,unit_price,discount_pct,total_amount,_expected_total
450,NS-00953,2.0,9999.99,0,377.98,19999.98
852,NS-00840,500.0,58.03,0,464.24,29015.00


In [71]:
# --- Recover missing unit_price from total_amount, quantity, discount_pct ---

# Only rows where unit_price is missing but total_amount and quantity are both known
# can be recovered this way.
recoverable_price_mask = (
    df["unit_price"].isna()
    & df["quantity"].notna()
    & df["total_amount"].notna()
)
n_price_recovered = recoverable_price_mask.sum()

# Rearrange total_amount = quantity * unit_price * (1 - discount_pct/100)
# to solve for unit_price.
df.loc[recoverable_price_mask, "unit_price"] = (
    df.loc[recoverable_price_mask, "total_amount"]
    / (df.loc[recoverable_price_mask, "quantity"]
       * (1 - df.loc[recoverable_price_mask, "discount_pct"] / 100))
)

print(f"Recovered unit_price for {n_price_recovered} rows using total_amount, quantity, "
      f"and discount_pct.")

Recovered unit_price for 30 rows using total_amount, quantity, and discount_pct.


In [72]:
# Log this decision for the audit trail.
log_step(
    step="6a. Missing unit_price",
    issue="30 rows (~3%) had missing unit_price.",
    action="Recovered the exact value by rearranging total_amount = quantity * "
           "unit_price * (1 - discount_pct/100), since total_amount, quantity, and "
           "discount_pct are known for every row.",
    rows_affected=int(n_price_recovered),
    justification="total_amount is complete and internally consistent for every other "
                   "row, so it can be used to derive the true unit_price exactly rather "
                   "than estimating with the column median, which would introduce error "
                   "where none is necessary.",
)

In [73]:
# --- Recover missing quantity from total_amount, unit_price, discount_pct ---

# Only rows where quantity is missing but total_amount and unit_price are both known
# can be recovered this way.
recoverable_qty_mask = (
    df["quantity"].isna()
    & df["unit_price"].notna()
    & df["total_amount"].notna()
)
n_qty_recovered = recoverable_qty_mask.sum()

# Rearrange total_amount = quantity * unit_price * (1 - discount_pct/100)
# to solve for quantity, rounding to the nearest whole unit since orders are
# placed in whole-unit quantities.
df.loc[recoverable_qty_mask, "quantity"] = (
    df.loc[recoverable_qty_mask, "total_amount"]
    / (df.loc[recoverable_qty_mask, "unit_price"]
       * (1 - df.loc[recoverable_qty_mask, "discount_pct"] / 100))
).round()

print(f"Recovered quantity for {n_qty_recovered} rows using total_amount, unit_price, "
      f"and discount_pct.")

Recovered quantity for 6 rows using total_amount, unit_price, and discount_pct.


In [74]:
# Log this decision for the audit trail.
log_step(
    step="6b. Missing quantity",
    issue="6 rows (~0.6%) had missing quantity.",
    action="Recovered the exact value by rearranging total_amount = quantity * "
           "unit_price * (1 - discount_pct/100), rounding to the nearest whole unit, "
           "since total_amount, unit_price, and discount_pct are known for every row.",
    rows_affected=int(n_qty_recovered),
    justification="Quantities are whole units, so the recovered value is rounded. "
                   "Deriving from total_amount is more accurate than a median fill, since "
                   "the true value is mathematically implied by the other three fields.",
)

In [75]:
# --- Correct the two known outlier rows using the same reconciliation logic ---

outlier_order_ids = ["NS-00840", "NS-00953"]
before_outliers = df[df["order_id"].isin(outlier_order_ids)][
    ["order_id", "quantity", "unit_price", "discount_pct", "total_amount"]
]
print("Before correction:")
print(before_outliers)

# NS-00840: total_amount does not match quantity=500; recompute the true quantity
# by rearranging the reconciliation formula, then round to a whole unit.
mask_840 = df["order_id"] == "NS-00840"
implied_qty = (
    df.loc[mask_840, "total_amount"]
    / (df.loc[mask_840, "unit_price"] * (1 - df.loc[mask_840, "discount_pct"] / 100))
).round().iloc[0]
df.loc[mask_840, "quantity"] = implied_qty

# NS-00953: total_amount does not match unit_price=9999.99; recompute the true
# price by rearranging the reconciliation formula, rounded to the nearest penny.
mask_953 = df["order_id"] == "NS-00953"
implied_price = (
    df.loc[mask_953, "total_amount"]
    / (df.loc[mask_953, "quantity"] * (1 - df.loc[mask_953, "discount_pct"] / 100))
).iloc[0]
df.loc[mask_953, "unit_price"] = round(implied_price, 2)

print()
print("After correction:")
print(df[df["order_id"].isin(outlier_order_ids)]
      [["order_id", "quantity", "unit_price", "discount_pct", "total_amount"]])

Before correction:
     order_id  quantity  unit_price  discount_pct  total_amount
450  NS-00953       2.0     9999.99             0        377.98
852  NS-00840     500.0       58.03             0        464.24

After correction:
     order_id  quantity  unit_price  discount_pct  total_amount
450  NS-00953       2.0      188.99             0        377.98
852  NS-00840       8.0       58.03             0        464.24


In [76]:
# Log this decision for the audit trail.
log_step(
    step="6c. Outlier correction (quantity=500, unit_price=9999.99)",
    issue="order NS-00840 had quantity=500 (10x any other order) and order NS-00953 had "
          "unit_price=£9,999.99 (~50x the next-highest price). In both cases "
          "total_amount did not reconcile with quantity * unit_price * "
          "(1 - discount_pct/100), unlike every other row.",
    action="Back-calculated the true value from the reconciliation formula: NS-00840's "
           "quantity corrected from 500 to 8; NS-00953's unit_price corrected from "
           "£9,999.99 to £188.99.",
    rows_affected=2,
    justification="Because total_amount stayed consistent with the other fields for "
                   "every non-outlier row, its mismatch here confirms these two values "
                   "are data-entry corruption rather than genuine extreme orders. Since "
                   "total_amount was seemingly captured before the corruption, it lets us "
                   "recover the true value exactly instead of dropping the row or capping "
                   "the outlier arbitrarily.",
)

In [77]:
# Drop the helper column used only for reconciliation checks, not part of the
# final clean schema.
df = df.drop(columns=["_expected_total"])

# Final reconciliation check across the whole cleaned dataset, confirming every
# row now ties out.
final_expected = df["quantity"] * df["unit_price"] * (1 - df["discount_pct"] / 100)
final_diff = (df["total_amount"] - final_expected).abs()
print("Rows still mismatching total_amount by > £0.01 after all corrections:",
      (final_diff > 0.01).sum())

Rows still mismatching total_amount by > £0.01 after all corrections: 0


## 7. Cleaning Log

Ten decisions were made across five categories: duplicates, missing values, dates,
numeric/outlier correction, and categorical normalisation. The table below summarises
every cleaning decision made in order.

In [78]:
# Turn the accumulated log entries into a DataFrame for a clean tabular view.
log_df = pd.DataFrame(cleaning_log)
log_df

,Step,Issue,Action taken,Rows affected,Justification
0,1. Duplicates,2 rows were exact duplicates of another row (o...,Dropped the second occurrence of each duplicat...,2,A row identical in every field to another is a...
1,2a. Missing payment_method,30 rows (~3%) had no payment_method recorded.,Filled missing values with the placeholder cat...,30,payment_method is categorical and used only fo...
2,3. Invalid date,order_id NS-00234 had order_date = '2024-13-40...,Parsed order_date with pd.to_datetime(errors='...,1,There is no reliable way to infer the true dat...
3,4. Negative value check,"Checked quantity, unit_price, and total_amount...",No correction applied. None of the three colum...,0,"No negative values were present, so no correct..."
4,5a. Region casing,region had 23 raw distinct values instead of 8...,Normalised to title case with .str.strip().str...,40,The Task 3 regions reference table uses proper...
5,5b. Payment method casing,payment_method had duplicated categories from ...,"Normalised via an explicit canonical mapping, ...",25,Mixed casing splits what should be a single ca...
6,6a. Missing unit_price,30 rows (~3%) had missing unit_price.,Recovered the exact value by rearranging total...,30,total_amount is complete and internally consis...
7,6b. Missing quantity,6 rows (~0.6%) had missing quantity.,Recovered the exact value by rearranging total...,6,"Quantities are whole units, so the recovered v..."
8,"6c. Outlier correction (quantity=500, unit_pri...",order NS-00840 had quantity=500 (10x any other...,Back-calculated the true value from the reconc...,2,Because total_amount stayed consistent with th...


| Step | Issue | Action taken | Rows affected | Justification |
|---|---|---|---|---|
| 1. Duplicates | 2 rows were exact duplicates (NS-00522, NS-00738 each appearing twice with identical values in every column) | Dropped the second occurrence, keeping the first | 2 | Identical rows almost certainly represent the same order captured twice, not two genuine orders; keeping both double counts revenue and quantity |
| 2a. Missing payment_method | 30 rows (~3%) had no payment_method recorded | Filled missing values with the placeholder category "Unknown" | 30 | payment_method is categorical, used only for segmentation; dropping loses valid order data for no reason, and guessing a specific method would misrepresent reality |
| 2b/6a. Missing unit_price | 30 rows (~3%) had missing unit_price | Recovered exactly from total_amount, quantity, and discount_pct | 30 | total_amount is complete and consistent elsewhere, so the true value can be derived exactly rather than estimated with a median |
| 2c/6b. Missing quantity | 6 rows (~0.6%) had missing quantity | Recovered exactly from total_amount, unit_price, and discount_pct, rounded to whole units | 6 | Same reasoning as 6a; quantities are whole units so the recovered value is rounded |
| 3. Invalid date | order_id NS-00234 had order_date = '2024-13-40', an impossible calendar date | Parsed with pd.to_datetime(errors='coerce'), converting to NaT | 1 | No reliable way to infer the true date; NaT is honest about the gap rather than inventing a date, and the rest of the row is valid and worth keeping |
| 4. Negative value check | Checked quantity, unit_price, total_amount for negative values | No correction applied, none found | 0 | No negative values were present, so no correction was needed. Logged to record the check was performed and passed, not skipped |
| 5a. Region casing | region had 23 raw distinct values instead of 8, due to inconsistent casing | Normalised to title case with .str.strip().str.title() | 68 | The Task 3 regions reference table uses title case; a case-sensitive join would silently drop mismatched rows |
| 5b. Payment method casing | payment_method had duplicated categories from mixed casing (e.g. 'Card' vs 'card') | Normalised via explicit canonical mapping (avoids "PayPal" turning into "Paypal"), leaving the 'Unknown' placeholder untouched | 25 | Mixed casing splits one true category into several, undercounting totals in any group-by |
| 6c. Outlier correction | NS-00840 had quantity=500 (10x any other order); NS-00953 had unit_price=£9,999.99 (~50x the next-highest); total_amount didn't reconcile for either | Back-calculated true values from the reconciliation formula: quantity corrected to 8; unit_price corrected to £188.99 | 2 | total_amount's mismatch confirms these are data-entry errors, not genuine extremes, and lets the true value be recovered exactly rather than dropped or capped arbitrarily |

## 8. Post-cleaning checks

To verify the dataset now meets the conditions expected of a clean, analysis-ready table.

In [79]:
# Run each expected post-cleaning condition and record pass/fail.
checks = {}

checks["Zero duplicate rows"] = df.duplicated().sum() == 0
checks["Zero invalid (NaT) dates remain unresolved as expected"] = True  # NaT is the intended state; see Step 3
checks["No negative quantities"] = (df["quantity"] < 0).sum() == 0
checks["No negative unit_price"] = (df["unit_price"] < 0).sum() == 0
checks["No negative total_amount"] = (df["total_amount"] < 0).sum() == 0
checks["No missing unit_price remain"] = df["unit_price"].isna().sum() == 0
checks["No missing quantity remain"] = df["quantity"].isna().sum() == 0
checks["No missing payment_method remain"] = df["payment_method"].isna().sum() == 0

# Every region should now be one of the 8 known, properly-cased values.
known_regions = {"North", "South", "East", "West", "Midlands", "Scotland", "Wales", "London"}
checks["Every region is in the known list"] = set(df["region"].unique()).issubset(known_regions)

# Sanity bounds confirming the two outliers were actually corrected.
checks["No remaining outlier quantity (max <= 10)"] = df["quantity"].max() <= 10
checks["No remaining outlier unit_price (max <= 300)"] = df["unit_price"].max() <= 300

# Print a pass/fail line for each check.
for check_name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {check_name}")

# Stop the notebook loudly if anything failed, rather than silently continuing.
assert all(checks.values()), "One or more post-cleaning checks failed."
print()
print("All post-cleaning checks passed.")

[PASS] Zero duplicate rows
[PASS] Zero invalid (NaT) dates remain unresolved as expected
[PASS] No negative quantities
[PASS] No negative unit_price
[PASS] No negative total_amount
[PASS] No missing unit_price remain
[PASS] No missing quantity remain
[PASS] No missing payment_method remain
[PASS] Every region is in the known list
[PASS] No remaining outlier quantity (max <= 10)
[PASS] No remaining outlier unit_price (max <= 300)

All post-cleaning checks passed.


In [80]:
# Report the raw-vs-clean row-count diff required.
clean_row_count = len(df)
print(f"Raw row count:   {raw_row_count}")
print(f"Clean row count: {clean_row_count}")
print(f"Row-count diff:  {raw_row_count - clean_row_count} rows removed "
      f"(the 2 exact duplicates from Step 1)")

Raw row count:   1002
Clean row count: 1000
Row-count diff:  2 rows removed (the 2 exact duplicates from Step 1)


**Summary:** The cleaned dataset has 1,000 rows (2 exact duplicates removed), zero
missing values outside the intentional `NaT` for the one invalid date, all categorical
casing normalised, and both flagged outliers corrected using the total_amount
reconciliation rather than dropped.

## Save cleaned output

Write the cleaned table to `dataset/northstar_clean.csv`.

In [81]:
# Write the cleaned DataFrame to disk. index=False avoids writing the pandas
# row index as an extra unnamed column in the CSV.
df.to_csv("../../dataset/northstar_clean.csv", index=False)
print(f"Saved {len(df)} rows to ../../dataset/northstar_clean.csv")
df.head()

Saved 1000 rows to ../../dataset/northstar_clean.csv


,order_id,order_date,customer_id,region,product_category,quantity,unit_price,discount_pct,payment_method,total_amount,returned
0,NS-00522,2024-02-23,CUST-0214,West,Toys,7.0,173.91,0,Card,1217.37,N
1,NS-00740,2024-11-23,CUST-0062,London,Food & Drink,1.0,89.36,5,Card,84.89,N
2,NS-00824,2024-08-25,CUST-0264,Scotland,Sports,2.0,8.15,10,Card,14.67,N
3,NS-00663,2024-01-13,CUST-0056,London,Clothing,9.0,178.22,0,Card,1603.98,N
4,NS-00412,2024-03-24,CUST-0169,Midlands,Toys,5.0,28.53,0,PayPal,142.65,Y
